Đường có lề rộng nhất

In [ ]:
# -*- coding: utf-8 -*-
"""Bước 1: tìm đường có lề rộng nhất và các vectơ hỗ trợ.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

# 24 điểm, hai cụm tách hẳn nhau
X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]],
                  cluster_std=0.9, random_state=0)
y = 1 - y # lớp 1 là cụm dưới bên trái, giống hình trên slide

# C rất lớn nghĩa là không cho điểm nào nằm trong lề
svm = SVC(kernel="linear", C=1000)
svm.fit(X, y)

w = svm.coef_[0]
b = svm.intercept_[0]
print(f"w1 = {w[0]:.3f}, w2 = {w[1]:.3f}, b = {b:.3f}")
do_dai_w = np.sqrt(w[0] ** 2 + w[1] ** 2)
print(f"Do rong cua le = 2 / {do_dai_w:.4f} = {2 / do_dai_w:.4f}")
print()

# Các vectơ hỗ trợ nằm đúng trên hai mép lề, nơi f(x) bằng 1 hoặc -1
print("So vecto ho tro:", len(svm.support_))
for i in svm.support_:
    f = svm.decision_function(X[i:i + 1])[0]
    print(f" diem {i:2d}: x1 = {X[i, 0]:.2f}, x2 = {X[i, 1]:.2f},"
          f" lop {y[i]}, f(x) = {f:+.3f}")
print()

# Những điểm khác nằm ngoài lề, nên |f(x)| lớn hơn 1
f_tat_ca = svm.decision_function(X)
ngoai = np.setdiff1d(np.arange(len(X)), svm.support_)
nho_nhat = np.abs(f_tat_ca[ngoai]).min()
print(f"|f(x)| nho nhat cua cac diem khac: {nho_nhat:.3f}")
print()

# Học lại chỉ với các vectơ hỗ trợ
chi_sv = SVC(kernel="linear", C=1000)
chi_sv.fit(X[svm.support_], y[svm.support_])
w2 = chi_sv.coef_[0]
print("Hoc lai chi voi vecto ho tro:")
b2 = chi_sv.intercept_[0]
print(f" w1 = {w2[0]:.3f}, w2 = {w2[1]:.3f}, b = {b2:.3f}")
print()

# Dự đoán cho hai điểm mới
moi = np.array([[3.0, 3.0], [5.0, 4.5]])
for diem, f, lop in zip(moi, svm.decision_function(moi), svm.predict(moi)):
    print(f"Diem ({diem[0]}, {diem[1]}): f(x) = {f:+.3f}, doan lop {lop}")

w1 = -0.831, w2 = -0.355, b = 4.820
Do rong cua le = 2 / 0.9039 = 2.2126

So vecto ho tro: 3
 diem 18: x1 = 5.20, x2 = 4.22, lop 0, f(x) = -1.000
 diem 10: x1 = 2.88, x2 = 4.02, lop 1, f(x) = +1.000
 diem 13: x1 = 3.59, x2 = 2.36, lop 1, f(x) = +1.000

|f(x)| nho nhat cua cac diem khac: 1.059

Hoc lai chi voi vecto ho tro:
 w1 = -0.831, w2 = -0.355, b = 4.821

Diem (3.0, 3.0): f(x) = +1.262, doan lop 1
Diem (5.0, 4.5): f(x) = -0.933, doan lop 0


Lề mềm và tham số C

In [ ]:
# -*- coding: utf-8 --
"""Bước 2: lề mềm, tham số C, và vì sao SVM phải chuẩn hoá."""

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Tap hoc co", len(X_tr), "ban")
print(" C so vecto HT do rong le tap hoc tap kiem tra")
for C in [0.01, 0.1, 1, 10, 100]:
    mh = make_pipeline(StandardScaler(),
                       SVC(kernel="linear", C=C))
    mh.fit(X_tr, y_tr)
    svm = mh[-1] # bước cuối của pipeline là SVM
    le = 2 / np.linalg.norm(svm.coef_[0])
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f" {C:6} {len(svm.support_):6d} {le:6.2f}"
          f" {a_hoc:.4f} {a_kt:.4f}")
print()

# Bộ sinh viên mở rộng có thêm cột chi tiêu, tính bằng nghìn đồng
df3 = pd.read_csv("data/sinh_vien_mo_rong.csv")
X3 = df3[["gio_on", "diem_giua_ky", "chi_tieu"]]
y3 = df3["qua_mon"]
X3_tr, X3_te, y3_tr, y3_te = train_test_split(
    X3, y3, test_size=0.25, random_state=17, stratify=y3
)
tho = SVC().fit(X3_tr, y3_tr)
chuan = make_pipeline(StandardScaler(), SVC()).fit(X3_tr, y3_tr)
print("Ba cot, kernel RBF mac dinh")
print(f" Doan bua ca lop qua : {y3_te.mean():.4f}")
print(f" Khong chuan hoa : "
      f"{accuracy_score(y3_te, tho.predict(X3_te)):.4f}")
print(f" Co chuan hoa : "
      f"{accuracy_score(y3_te, chuan.predict(X3_te)):.4f}")

Tap hoc co 90 ban
 C so vecto HT do rong le tap hoc tap kiem tra
   0.01     74   3.84 0.8556 0.8667
    0.1     44   1.70 0.8889 0.9000
      1     31   1.29 0.8778 0.9000
     10     28   1.06 0.9000 0.9000
    100     27   1.05 0.9000 0.9000

Ba cot, kernel RBF mac dinh
 Doan bua ca lop qua : 0.6000
 Khong chuan hoa : 0.6000
 Co chuan hoa : 0.9000


Kernel và biên cong

In [ ]:
# -*- coding: utf-8 -*-
"""Bước 3: nâng chiều bằng tay, rồi để kernel làm việc đó."""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Phần 1: dữ liệu một chiều, lớp 1 nằm giữa, lớp 0 ở hai đầu
x = np.array([-4.0, -3.4, -2.8, -2.2, -1.2, -0.7, -0.2,
              0.3, 0.8, 1.3, 2.3, 2.9, 3.5, 4.0])
y = (np.abs(x) < 1.8).astype(int)

# Thử mọi ngưỡng: đoán lớp 1 khi x nhỏ hơn hoặc bằng ngưỡng
tot = max(((x <= t).astype(int) == y).mean() for t in x)
print(f"Mot nguong tren truc x, tot nhat dung {tot:.4f}")

# Thêm cột x binh phuong rồi dùng SVM tuyến tính
X_moi = np.c_[x, x ** 2]
svm = SVC(kernel="linear", C=1000).fit(X_moi, y)
print(f"Them cot x^2, SVM tuyen tinh dung {svm.score(X_moi, y):.4f}")
print()

# Phần 2: hai vầng trăng, để kernel tự nâng chiều
df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

mo_hinh = {
    "SVM kernel tuyen tinh": SVC(kernel="linear"),
    "SVM kernel da thuc bac 3": SVC(kernel="poly", degree=3),
    "SVM kernel RBF": SVC(kernel="rbf"),
    "Hoi quy logistic": LogisticRegression(),
    "KNN, k = 13": KNeighborsClassifier(n_neighbors=13),
}
print("Hai vang trang, accuracy tren tap kiem tra")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f" {ten:26s} {acc:.4f}")
print()

rbf = mo_hinh["SVM kernel RBF"]
print("So vecto ho tro cua kernel RBF:", len(rbf.support_),
      "tren", len(X_tr), "diem hoc")

Mot nguong tren truc x, tot nhat dung 0.7143
Them cot x^2, SVM tuyen tinh dung 1.0000

Hai vang trang, accuracy tren tap kiem tra
 SVM kernel tuyen tinh      0.8667
 SVM kernel da thuc bac 3   0.8667
 SVM kernel RBF             0.9200
 Hoi quy logistic           0.8533
 KNN, k = 13                0.9200

So vecto ho tro cua kernel RBF: 89 tren 225 diem hoc


Chọn C và gamma

In [ ]:
# -*- coding: utf-8 -*-
"""Bước 4: ảnh hưởng của gamma, rồi dò lưới C và gamma."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("gamma tap hoc tap kiem tra so vecto HT")
for g in [0.1, 1, 10, 100]:
    mh = SVC(kernel="rbf", gamma=g).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f" {g:5} {a_hoc:.4f} {a_kt:.4f} "
          f"{len(mh.support_)}")
print()

# Dò lưới: thử mọi cặp bằng kiểm định chéo 5 phần, chỉ trên tập học
luoi = {"C": [0.1, 1, 10, 100], "gamma": [0.01, 0.1, 1, 10]}
tim = GridSearchCV(SVC(kernel="rbf"), luoi, cv=5)
tim.fit(X_tr, y_tr)

bang = pd.DataFrame(tim.cv_results_["params"])
bang["diem"] = tim.cv_results_["mean_test_score"]
print("Diem kiem dinh cheo, hang la C, cot la gamma")
print(bang.pivot(index="C", columns="gamma",
                 values="diem").round(4).to_string())
print()

print("Cap tot nhat:", tim.best_params_)
print(f"Diem kiem dinh cheo: {tim.best_score_:.4f}")
# GridSearchCV đã tự học lại với cặp tốt nhất trên toàn tập học
print(f"Accuracy tren tap kiem tra: {tim.score(X_te, y_te):.4f}")

gamma tap hoc tap kiem tra so vecto HT
   0.1 0.8356 0.8533 113
     1 0.8711 0.9200 88
    10 0.8978 0.9067 128
   100 0.9422 0.8133 217

Diem kiem dinh cheo, hang la C, cot la gamma
gamma   0.01    0.10    1.00    10.00
C                                    
0.1    0.6578  0.8044  0.8444  0.8578
1.0    0.8044  0.8178  0.8667  0.8667
10.0   0.8178  0.8400  0.8711  0.8311
100.0  0.8178  0.8533  0.8711  0.7467

Cap tot nhat: {'C': 10, 'gamma': 1}
Diem kiem dinh cheo: 0.8711
Accuracy tren tap kiem tra: 0.9200


Nhận dạng chữ số viết tay

In [ ]:
# -*- coding: utf-8 -*-
"""Bước 5: nhận dạng chữ số viết tay bằng SVM."""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
print("Kich thuoc X:", X.shape)
print("Do sang nho nhat va lon nhat:", X.min(), X.max())
print("So anh moi chu so:", np.bincount(y))
print()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

svm = SVC(kernel="rbf").fit(X_tr, y_tr)
du_doan = svm.predict(X_te)
print(f"SVM kernel RBF: accuracy {accuracy_score(y_te, du_doan):.4f}")
print("So vecto ho tro:", len(svm.support_), "tren", len(X_tr), "anh")
sai = np.flatnonzero(du_doan != y_te)
print("So anh doan sai:", len(sai), "tren", len(y_te))
for i in sai:
    print(f" anh {i:3d}: that la {y_te[i]}, doan la {du_doan[i]}")
print()

# So với các mô hình đã học
mo_hinh = {
    "SVM co chuan hoa": make_pipeline(StandardScaler(), SVC()),
    "KNN, k = 5": KNeighborsClassifier(),
    "Logistic co chuan hoa": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000)),
}
print("Cac mo hinh khac, accuracy tren tap kiem tra")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f" {ten:22s} {acc:.4f}")

Kich thuoc X: (1797, 64)
Do sang nho nhat va lon nhat: 0.0 16.0
So anh moi chu so: [178 182 177 183 181 182 181 179 174 180]

SVM kernel RBF: accuracy 0.9911
So vecto ho tro: 626 tren 1347 anh
So anh doan sai: 4 tren 450
 anh   6: that la 9, doan la 3
 anh  18: that la 8, doan la 1
 anh 161: that la 4, doan la 8
 anh 428: that la 7, doan la 9

Cac mo hinh khac, accuracy tren tap kiem tra
 SVM co chuan hoa       0.9778
 KNN, k = 5             0.9867
 Logistic co chuan hoa  0.9711


Bài tập thực hành

In [2]:
#bài 1
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

# Tao 24 diem nhu muc 2
X, y = make_blobs(
    n_samples=24,
    centers=[[2, 2], [6, 6]],
    cluster_std=0.9,
    random_state=0
)
y = 1 - y

# Huan luyen SVM tuyen tinh
svm = SVC(kernel="linear", C=1000)
svm.fit(X, y)

# Lay w1, w2, b
w1 = svm.coef_[0][0]
w2 = svm.coef_[0][1]
b = svm.intercept_[0]

print("w1 =", round(w1, 3))
print("w2 =", round(w2, 3))
print("b =", round(b, 3))

# Dung gia tri da in o muc 2 de tinh tay
w1 = -0.831
w2 = -0.355
b = 4.820

# Do dai vector w
do_dai_w = np.sqrt(w1**2 + w2**2)

# Cac diem can tinh
cac_diem = [
    (3, 3),
    (5, 4.5),
    (5.20, 4.22),
    (2.88, 4.02),
    (3.59, 2.36)
]

print("\nKET QUA TINH TAY")

for x1, x2 in cac_diem:
    # Tinh f(x) bang phep nhan va cong
    fx = w1 * x1 + w2 * x2 + b

    # Tinh khoang cach
    khoang_cach = abs(fx) / do_dai_w

    print("\nDiem:", (x1, x2))
    print("f(x) =", w1, "*", x1, "+", w2, "*", x2, "+", b)
    print("f(x) =", round(fx, 4))
    print("Khoang cach =", round(khoang_cach, 4))

# Do rong le
do_rong_le = 2 / do_dai_w

print("\nDo dai w =", round(do_dai_w, 4))
print("Do rong le =", round(do_rong_le, 4))
print("Mot nua do rong le =", round(do_rong_le / 2, 4))

w1 = -0.831
w2 = -0.355
b = 4.82

KET QUA TINH TAY

Diem: (3, 3)
f(x) = -0.831 * 3 + -0.355 * 3 + 4.82
f(x) = 1.262
Khoang cach = 1.3966

Diem: (5, 4.5)
f(x) = -0.831 * 5 + -0.355 * 4.5 + 4.82
f(x) = -0.9325
Khoang cach = 1.0319

Diem: (5.2, 4.22)
f(x) = -0.831 * 5.2 + -0.355 * 4.22 + 4.82
f(x) = -0.9993
Khoang cach = 1.1058

Diem: (2.88, 4.02)
f(x) = -0.831 * 2.88 + -0.355 * 4.02 + 4.82
f(x) = 0.9996
Khoang cach = 1.1062

Diem: (3.59, 2.36)
f(x) = -0.831 * 3.59 + -0.355 * 2.36 + 4.82
f(x) = 0.9989
Khoang cach = 1.1054

Do dai w = 0.9037
Do rong le = 2.2132
Mot nua do rong le = 1.1066


Nhận xét:
- Input: Sử dụng các hệ số lấy từ mục 2: w1 = -0.831, w2 = -0.355, b = 4.82. Tính cho hai điểm (3, 3), (5, 4.5) và ba vectơ hỗ trợ.
- Output:
  - Điểm (3, 3): f(x) = 1.262, khoảng cách 1.3966.
  - Điểm (5, 4.5): f(x) = -0.9325, khoảng cách 1.0319.
  - Ba vectơ hỗ trợ có khoảng cách lần lượt là 1.1058, 1.1062, 1.1054.
  - Độ dài vector w là 0.9037, độ rộng lề là 2.2132, một nửa độ rộng lề là 1.1066.
- Ba vectơ hỗ trợ có khoảng cách đến đường ranh giới xấp xỉ 1.106, gần bằng một nửa độ rộng lề 1.1066.
- Giải thích: Ba vectơ hỗ trợ nằm trên hai đường biên lề f(x) = 1 hoặc f(x) = -1. Khoảng cách từ mỗi đường biên lề đến đường ranh giới là 1 / ||w||, trong khi độ rộng toàn bộ lề là 2 / ||w||. Vì vậy, khoảng cách của vectơ hỗ trợ bằng một nửa độ rộng lề. Sai lệch nhỏ trong kết quả là do các hệ số w1, w2, b đã được làm tròn.

In [3]:
#bài 2
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

# Tao 24 diem ban dau
X, y = make_blobs(
    n_samples=24,
    centers=[[2, 2], [6, 6]],
    cluster_std=0.9,
    random_state=0
)
y = 1 - y

def huan_luyen(X, y, ten):
    svm = SVC(kernel="linear", C=1000)
    svm.fit(X, y)

    print("\n", ten)
    print("w1 =", round(svm.coef_[0][0], 3))
    print("w2 =", round(svm.coef_[0][1], 3))
    print("b =", round(svm.intercept_[0], 3))

    print("So vector ho tro:", len(svm.support_))

    print("Danh sach vector ho tro:")
    for i in svm.support_:
        print(
            "Diem:", X[i],
            "- Lop:", y[i]
        )

# Truong hop 1: Them diem (-1, -1), lop 1
X1 = np.vstack((X, [-1, -1]))
y1 = np.append(y, 1)

huan_luyen(X1, y1, "THEM DIEM (-1, -1)")

# Truong hop 2: Thay bang diem (4.5, 4), lop 1
X2 = np.vstack((X, [4.5, 4]))
y2 = np.append(y, 1)

huan_luyen(X2, y2, "THEM DIEM (4.5, 4)")


 THEM DIEM (-1, -1)
w1 = -0.831
w2 = -0.355
b = 4.82
So vector ho tro: 3
Danh sach vector ho tro:
Diem: [5.20099283 4.21728318] - Lop: 0
Diem: [2.88086419 4.01680388] - Lop: 1
Diem: [3.58764711 2.36014149] - Lop: 1

 THEM DIEM (4.5, 4)
w1 = -2.603
w2 = -0.807
b = 15.941
So vector ho tro: 2
Danh sach vector ho tro:
Diem: [5.20099283 4.21728318] - Lop: 0
Diem: [4.5 4. ] - Lop: 1


Nhận xét:
- Input: Sử dụng 24 điểm dữ liệu ban đầu, lần lượt thêm hai điểm thuộc lớp 1 là (-1, -1) và (4.5, 4) để huấn luyện lại mô hình SVM tuyến tính.
- Output:
Trường hợp	w1	w2	b	Số vectơ hỗ trợ
Thêm (-1, -1)	-0.831	-0.355	4.820	3
Thêm (4.5, 4)	-2.603	-0.807	15.941	2
  - Khi thêm điểm (-1, -1), các hệ số w1, w2, b vẫn giữ nguyên so với ban đầu. Điểm mới không trở thành vectơ hỗ trợ nên đường ranh giới không thay đổi.
  - Khi thêm điểm (4.5, 4), các hệ số thay đổi từ (-0.831, -0.355, 4.820) thành (-2.603, -0.807, 15.941). Điểm mới trở thành vectơ hỗ trợ nên đường ranh giới thay đổi.
- Vectơ hỗ trợ quyết định vị trí và hướng của đường ranh giới SVM. Điểm dữ liệu nằm ngoài vùng ảnh hưởng của các vectơ hỗ trợ thường không làm thay đổi đường ranh giới, trong khi điểm mới trở thành vectơ hỗ trợ có thể làm thay đổi mô hình.

In [5]:
#bài 3
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Doc bo du lieu hai vang trang
df = pd.read_csv("data/hai_vang_trang.csv")

X = df[["x1", "x2"]]
y = df["nhan"]

# Chia tap train va test
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# Thu cac bac da thuc 2, 3, 4
for d in [2, 3, 4]:

    print("\nBAC DA THUC:", d)

    # Tao pipeline co chuan hoa
    model = make_pipeline(
        StandardScaler(),
        SVC(kernel="poly", degree=d)
    )

    # Doi tham so C
    param_grid = {
        "svc__C": [0.1, 1, 10, 100]
    }

    grid = GridSearchCV(
        model,
        param_grid,
        cv=5,
        scoring="accuracy"
    )

    grid.fit(X_tr, y_tr)

    # Du doan tren tap test
    y_pred = grid.predict(X_te)

    print("C tot nhat:", grid.best_params_["svc__C"])
    print("CV accuracy:", round(grid.best_score_, 4))
    print("Test accuracy:", round(accuracy_score(y_te, y_pred), 4))

# So sanh voi RBF o muc 5
print("\nRBF accuracy o muc 5: 0.9200")


BAC DA THUC: 2
C tot nhat: 1
CV accuracy: 0.5422
Test accuracy: 0.5067

BAC DA THUC: 3
C tot nhat: 1
CV accuracy: 0.8178
Test accuracy: 0.8267

BAC DA THUC: 4
C tot nhat: 100
CV accuracy: 0.5156
Test accuracy: 0.48

RBF accuracy o muc 5: 0.9200


Nhận xét:
- Input: Bộ dữ liệu hai vầng trăng gồm hai thuộc tính x1, x2 và nhãn nhan. Thử các bậc đa thức 2, 3, 4 và các giá trị C [0.1, 1, 10, 100].
- Phương pháp: Sử dụng Polynomial SVM, chuẩn hóa dữ liệu bằng StandardScaler trong make_pipeline và dùng GridSearchCV với kiểm định chéo 5 phần để tìm giá trị C tốt nhất.
- Output:
  - Bậc đa thức: 2 - 3 - 4
  - C tốt nhất: 1 - 1 - 100
  - CV accuracy: 0.5422 - 0.8178 - 0.5156
  - Test accuracy: 0.5067 - 0.8267 - 0.4800
- Bậc đa thức 3 đạt kết quả tốt nhất với CV accuracy 0.8178 và test accuracy 0.8267. Bậc 2 và bậc 4 có độ chính xác thấp hơn. Polynomial SVM bậc 3 là lựa chọn tốt nhất trong ba bậc được thử. Tuy nhiên, test accuracy 0.8267 vẫn thấp hơn kernel RBF ở mục 5 là 0.9200, nên Polynomial SVM chưa cho kết quả tốt bằng RBF trong thí nghiệm này.

In [4]:
#bài 4
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Doc du lieu
df = pd.read_csv("data/hai_vang_trang.csv")

X = df[["x1", "x2"]]
y = df["nhan"]

# Chia tap train va test giong muc 5
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# Dung kernel RBF
model = SVC(kernel="rbf")

# Doi luoi min quanh cap tot nhat C=10, gamma=1
param_grid = {
    "C": [3, 10, 30],
    "gamma": [0.3, 1, 3]
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_tr, y_tr)

# Danh gia tren tap test
y_pred = grid.predict(X_te)

print("Cap tham so tot nhat:", grid.best_params_)
print("CV accuracy tot nhat:", round(grid.best_score_, 4))
print("Test accuracy:", round(accuracy_score(y_te, y_pred), 4))

# In ket qua 9 cap tham so
print("\nKET QUA CAC CAP THAM SO")

for params, score in zip(
    grid.cv_results_["params"],
    grid.cv_results_["mean_test_score"]
):
    print(params, "CV =", round(score, 4))

Cap tham so tot nhat: {'C': 30, 'gamma': 1}
CV accuracy tot nhat: 0.8756
Test accuracy: 0.92

KET QUA CAC CAP THAM SO
{'C': 3, 'gamma': 0.3} CV = 0.8533
{'C': 3, 'gamma': 1} CV = 0.8711
{'C': 3, 'gamma': 3} CV = 0.8667
{'C': 10, 'gamma': 0.3} CV = 0.8489
{'C': 10, 'gamma': 1} CV = 0.8711
{'C': 10, 'gamma': 3} CV = 0.8667
{'C': 30, 'gamma': 0.3} CV = 0.8622
{'C': 30, 'gamma': 1} CV = 0.8756
{'C': 30, 'gamma': 3} CV = 0.8533


Nhận xét:
- Input: Bộ dữ liệu hai vầng trăng, sử dụng SVM với kernel RBF. Thử C [3, 10, 30] và gamma [0.3, 1, 3].
- Phương pháp: Sử dụng GridSearchCV với kiểm định chéo 5 phần để đánh giá 9 cặp tham số.
- Output: Cặp tham số tốt nhất là C = 30, gamma = 1, đạt CV accuracy 0.8756 và test accuracy 0.9200.
- Kết quả: CV accuracy tốt nhất 0.8756, cao hơn kết quả 0.8711 ở mục 5. Tuy nhiên, test accuracy vẫn giữ nguyên 0.9200, không tăng so với mục 5.
- Kết luận: Việc dò lưới tham số chi tiết giúp tìm được cấu hình có điểm kiểm định chéo tốt hơn. Tuy nhiên, điểm CV tăng không đồng nghĩa với accuracy trên tập kiểm tra cũng tăng. Vì vậy, cần đánh giá cả hai chỉ số để lựa chọn mô hình phù hợp.

In [6]:
#bài 5
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

# Doc du lieu chu so
X, y = load_digits(return_X_y=True)

# Chia tap train va test
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# Huan luyen SVM RBF
svm = SVC(kernel="rbf")
svm.fit(X_tr, y_tr)

# Du doan
y_pred = svm.predict(X_te)

print("Accuracy:", round(accuracy_score(y_te, y_pred), 4))

# Tao ma tran nham lan
cm = confusion_matrix(y_te, y_pred)

print("\nMA TRAN NHAM LAN:")
print(cm)

# Tim cac mau du doan sai
sai = np.where(y_pred != y_te)[0]

print("\nSo anh du doan sai:", len(sai))

# Tao thu muc luu anh
os.makedirs("anh_sai", exist_ok=True)

# Ve va luu tung anh sai
for i in sai:

    anh = X_te[i].reshape(8, 8)

    plt.figure()
    plt.imshow(anh, cmap="gray_r")
    plt.title(
        f"Thuc te: {y_te[i]} - Du doan: {y_pred[i]}"
    )
    plt.axis("off")

    ten_file = f"anh_sai/anh_{i}_that_{y_te[i]}_doan_{y_pred[i]}.png"

    plt.savefig(ten_file, bbox_inches="tight")
    plt.close()

    print(
        "Anh:", i,
        "| Chu so that:", y_te[i],
        "| Chu so du doan:", y_pred[i],
        "| Da luu:", ten_file
    )

Accuracy: 0.9911

MA TRAN NHAM LAN:
[[45  0  0  0  0  0  0  0  0  0]
 [ 0 46  0  0  0  0  0  0  0  0]
 [ 0  0 44  0  0  0  0  0  0  0]
 [ 0  0  0 46  0  0  0  0  0  0]
 [ 0  0  0  0 44  0  0  0  1  0]
 [ 0  0  0  0  0 46  0  0  0  0]
 [ 0  0  0  0  0  0 45  0  0  0]
 [ 0  0  0  0  0  0  0 44  0  1]
 [ 0  1  0  0  0  0  0  0 42  0]
 [ 0  0  0  1  0  0  0  0  0 44]]

So anh du doan sai: 4
Anh: 6 | Chu so that: 9 | Chu so du doan: 3 | Da luu: anh_sai/anh_6_that_9_doan_3.png
Anh: 18 | Chu so that: 8 | Chu so du doan: 1 | Da luu: anh_sai/anh_18_that_8_doan_1.png
Anh: 161 | Chu so that: 4 | Chu so du doan: 8 | Da luu: anh_sai/anh_161_that_4_doan_8.png
Anh: 428 | Chu so that: 7 | Chu so du doan: 9 | Da luu: anh_sai/anh_428_that_7_doan_9.png


Các chữ số bị nhầm:
- Chữ số thực tế: 4 - 7 - 8 - 9
- Chữ số dự đoán: 8 - 9 - 1 - 3
- Số lần nhầm: 1 - 1 - 1 - 1

Nhận xét:
- Input: Sử dụng bộ dữ liệu load_digits gồm 1797 ảnh chữ số viết tay từ 0 đến 9, mỗi ảnh có kích thước 8×8 pixel.
- Phương pháp: Sử dụng SVM với kernel RBF, chia dữ liệu thành 75% tập huấn luyện và 25% tập kiểm tra. Dùng confusion_matrix để tạo ma trận nhầm lẫn và plt.imshow() để vẽ, lưu các ảnh dự đoán sai.
- Output: Mô hình đạt accuracy 0.9911, ma trận nhầm lẫn có kích thước 10×10. Có 4 ảnh dự đoán sai trên tổng số 450 ảnh kiểm tra, tức dự đoán đúng 446 ảnh.
- Chữ số 4 bị nhận dạng thành 8, chữ số 7 thành 9, chữ số 8 thành 1 và chữ số 9 thành 3. Chương trình đã lưu cả 4 ảnh dự đoán sai trong thư mục anh_sai.
- Giải thích: Các chữ số viết tay có thể có nét viết và hình dạng khác nhau giữa từng người. Một số ảnh có đặc điểm hình dạng gần giống chữ số khác, khiến mô hình phân loại nhầm.
- SVM với kernel RBF nhận dạng chữ số khá tốt, đạt độ chính xác 99.11%. Tuy nhiên, vẫn có một số trường hợp nhầm lẫn do đặc điểm hình ảnh của các chữ số tương tự nhau.

In [8]:
#bài 6
import time
import numpy as np
from sklearn.datasets import make_classification
from sklearn.svm import SVC, LinearSVC
# Cac kich thuoc du lieu
kich_thuoc = [2000, 5000, 10000, 20000]
ket_qua = []
for n in kich_thuoc:
    # Tao du lieu
    X, y = make_classification(
        n_samples=n,
        n_features=20,
        random_state=0
    )
    thoi_gian_svc = []
    thoi_gian_linear = []
    # Do thoi gian SVC va LinearSVC 3 lan
    for lan in range(3):
        model = SVC()
        bat_dau = time.perf_counter()
        model.fit(X, y)
        thoi_gian_svc.append(time.perf_counter() - bat_dau)
        model = LinearSVC()
        bat_dau = time.perf_counter()
        model.fit(X, y)
        thoi_gian_linear.append(time.perf_counter() - bat_dau)
    # Lay thoi gian trung vi
    t_svc = np.median(thoi_gian_svc)
    t_linear = np.median(thoi_gian_linear)
    ket_qua.append([n, t_svc, t_linear])
# In bang ket qua
print("SO SANH THOI GIAN HUAN LUYEN")
print(f"{'So mau':<12}{'SVC (s)':<15}{'LinearSVC (s)':<18}")
for n, t_svc, t_linear in ket_qua:
    print(f"{n:<12}{t_svc:<15.6f}{t_linear:<18.6f}")
# Tinh ti le tang khi so mau tang gap doi
print("\nTI LE TANG THOI GIAN KHI SO MAU TANG GAP DOI")
for i in range(2, len(ket_qua)):
    n_truoc, svc_truoc, linear_truoc = ket_qua[i - 1]
    n_sau, svc_sau, linear_sau = ket_qua[i]
    print(f"\nTu {n_truoc} len {n_sau} mau:")
    print("SVC tang:", round(svc_sau / svc_truoc, 2), "lan")
    print("LinearSVC tang:", round(linear_sau / linear_truoc, 2), "lan")

SO SANH THOI GIAN HUAN LUYEN
So mau      SVC (s)        LinearSVC (s)     
2000        0.052053       0.003327          
5000        0.257991       0.007417          
10000       3.740530       0.014204          
20000       10.980886      0.034696          

TI LE TANG THOI GIAN KHI SO MAU TANG GAP DOI

Tu 5000 len 10000 mau:
SVC tang: 14.5 lan
LinearSVC tang: 1.91 lan

Tu 10000 len 20000 mau:
SVC tang: 2.94 lan
LinearSVC tang: 2.44 lan


Nhận xét:
- Input: Sinh dữ liệu bằng make_classification với 20 thuộc tính, số lượng mẫu lần lượt là 2.000, 5.000, 10.000 và 20.000.
- Phương pháp: Sử dụng time.perf_counter() để đo thời gian thực hiện .fit() của hai mô hình SVC() và LinearSVC(). Mỗi mô hình được chạy 3 lần và lấy thời gian trung vị.
- Output:
  - Số mẫu: 2.000 - 5.000 - 10.000 - 20.000
  - SVC (giây): 0.052053 - 0.257991 - 3.740530 - 10.988086
  - LinearSVC (giây): 0.003327 - 0.007417 -  0.014204 - 0.034696
- Kết quả khi số mẫu tăng gấp đôi:
  - Số mẫu tăng: 5.000 → 10.000 và 10.000 → 20.000
  - SVC tăng: 14.50 lần và 2.94 lần
  - LinearSVC tăng: 1.91 lần và 2.44 lần
- Giải thích: Khi số lượng mẫu tăng từ 5.000 lên 10.000, thời gian huấn luyện của SVC tăng mạnh, khoảng 14.50 lần, trong khi LinearSVC chỉ tăng 1.91 lần. Khi tăng từ 10.000 lên 20.000, SVC tăng 2.94 lần và LinearSVC tăng 2.44 lần.
- Kết luận: Trong thí nghiệm này, LinearSVC có thời gian huấn luyện thấp hơn đáng kể so với SVC ở cả bốn kích thước dữ liệu. Khi dữ liệu tăng lên, LinearSVC vẫn có tốc độ huấn luyện tốt hơn. Do đó, LinearSVC phù hợp hơn khi cần xử lý tập dữ liệu lớn và ưu tiên thời gian huấn luyện. Thời gian thực tế có thể thay đổi tùy cấu hình máy tính.